In [8]:
import torch
import numpy as np
import anndata as ad
from scipy.spatial.distance import pdist, squareform

from data.simulations import make_gaussian_to_moons
from models.autoencoder import NBAutoEncoder
from training.losses import LossComposer, NBReconLoss, PullbackIsotropyLoss, DistancePreservationLoss
from training.trainer_ae import make_ae_dataloader, train_ae, train_ae_two_phase

import plotly.graph_objects as go

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [16]:
adata = make_gaussian_to_moons(n_samples=1000)
adata.X = (adata.X - adata.X.min()) * 10

# save to h5ad
adata.write_h5ad("gaussian_to_moons.h5ad")

# adata.X = np.round(adata.X)
distances_np = squareform(pdist(adata.X))

train_loader = make_ae_dataloader(adata, distances=distances_np, batch_size=128, shuffle=True)

In [11]:
n_genes = adata.shape[1]
latent_dim = 2
model = NBAutoEncoder(n_genes=n_genes, latent_dim=latent_dim, hidden_dim=64, n_layers=3)

In [12]:
recon_loss = NBReconLoss()
isotropy_loss = PullbackIsotropyLoss()
distance_loss = DistancePreservationLoss()

loss_map = {
    "recon": recon_loss,
    "isotropy": isotropy_loss,
    "distance": distance_loss,
}

phase1_weights = {"recon": 0.0, "isotropy": 0.0, "distance": 1.0}
phase2_weights = {"recon": 1.0, "isotropy": 0.0, "distance": 0.0}

loss_composer = LossComposer(loss_map=loss_map, loss_weights=phase1_weights)

In [13]:
history = train_ae_two_phase(
    model=model,
    train_loader=train_loader,
    phase1_weights=phase1_weights,
    phase2_weights=phase2_weights,
    encoder_epochs=30,
    decoder_epochs=50,
    learning_rate=1e-3,
    device=device,
    gene_subsample=None,
    loss_composer=loss_composer,
)

starting phase 1: training encoder with distance-based losses...
training nb autoencoder...
encoder frozen: False, decoder frozen: True
loss terms: ['recon', 'isotropy', 'distance']


Epoch:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch: 100%|██████████| 30/30 [00:01<00:00, 18.47it/s, train_loss=2.4008, val_loss=N/A, train_recon=0.0000, train_isotropy=0.0000, train_distance=2.4008]   




starting phase 2: training decoder with reconstruction-based losses...
training nb autoencoder...
encoder frozen: True, decoder frozen: False
loss terms: ['recon', 'isotropy', 'distance']


Epoch: 100%|██████████| 50/50 [00:03<00:00, 16.03it/s, train_loss=3.4867, val_loss=N/A, train_recon=3.4867, train_isotropy=0.0000, train_distance=0.0000]


In [14]:
model.eval()
model.to(device)
with torch.no_grad():
    x_log_norm = torch.log1p(torch.tensor(adata.X, dtype=torch.float32).to(device))
    z = model.encoder(x_log_norm).cpu().numpy()

fig = go.Figure()
fig.add_trace(go.Scatter(x=adata.X[:, 0], y=adata.X[:, 1], mode="markers", name="Original"))
fig.add_trace(go.Scatter(x=z[:, 0], y=z[:, 1], mode="markers", name="Latent"))
fig.update_layout(title="Original vs Latent Space", xaxis_title="Dimension 1", yaxis_title="Dimension 2")
fig.show()

In [15]:
# save model and history
torch.save(model.state_dict(), "toy_ae_model.pt")